# SDTM Variable Identity — Pass B (3a-bis)

**Purpose.** Extend `SDTM_Variable_Identity.xlsx` with NCIt identity for
COSMoS variables that are missing from the two primary subsets (CDISC
Variable Terminology / CDISC Root Variable Terminology).

**Why.** Pass A covered 338/449 COSMoS variables. The remaining 111 have
no NCIt identity in those two subsets. They fall into three groups:
- (a) present in a different CDISC Variable Terminology subset (CDASH, SEND)
- (b) present in Thesaurus but with no Variable Terminology subset flag at all
- (c) absent from Thesaurus as an ALL-CAPS synonym.

This notebook finds (a) and (b); (c) remain unresolved and are reported.

**Inputs**
- `sdtm-narrative/reference/SDTM_Variable_Identity.xlsx` — Pass A output
- `sdtm-test-codes/downloads/Thesaurus.txt`
- `cosmos-graph/interim/COSMoS_Graph.xlsx`

**Output.** Same xlsx path — Pass A rows preserved, Pass B rows appended.
`Subset` column gets new enum values:
- `CDASH` — in CDISC CDASH Variable Terminology
- `SEND` — in CDISC SEND Variable Terminology
- `CDASH;SEND` — in both
- `Other` — in Thesaurus but in no CDISC Variable Terminology subset

**Disambiguation.** The variable code appears as an ALL-CAPS pipe-separated
synonym. Concepts are excluded if their preferred term looks like a codelist
container (starts with 'CDISC' and contains 'Terminology'). If multiple
concepts survive filtering, all are kept and flagged in `Notes`.


## Setup


In [ ]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPO = Path.cwd().parent.parent
THES     = REPO / 'sdtm-test-codes'  / 'downloads'  / 'Thesaurus.txt'
GRAPH    = REPO / 'cosmos-graph'     / 'interim'    / 'COSMoS_Graph.xlsx'
IDENTITY = REPO / 'sdtm-narrative'   / 'reference'  / 'SDTM_Variable_Identity.xlsx'

assert IDENTITY.exists(), f'Pass A output missing: {IDENTITY}'
print(f'Identity xlsx: {IDENTITY}')


## Load Pass A identity + COSMoS variable universe


In [ ]:
pass_a = pd.read_excel(IDENTITY, sheet_name='Variable_Identity', dtype=str, na_filter=False)
print(f'Pass A rows: {len(pass_a)}')

# COSMoS variables and their domains
graph_vars = pd.read_excel(GRAPH, sheet_name='Variables')[['ds_id', 'variable_name']]
graph_dss  = pd.read_excel(GRAPH, sheet_name='DSS')[['ds_id', 'domain']]
vars_with_domain = graph_vars.merge(graph_dss, on='ds_id', how='left')
cosmos_var_domains = (vars_with_domain
    .groupby('variable_name')['domain']
    .apply(lambda s: sorted(set(s.dropna())))
    .to_dict()
)
cosmos_variables = set(cosmos_var_domains.keys())

# Missing = in COSMoS but not covered by Pass A
pass_a_vars = set(pass_a.loc[pass_a['Variable'] != '', 'Variable'])
missing = sorted(cosmos_variables - pass_a_vars)
print(f'COSMoS variables:           {len(cosmos_variables)}')
print(f'Covered by Pass A:          {len(cosmos_variables & pass_a_vars)}')
print(f'Missing (Pass B targets):   {len(missing)}')


## Load Thesaurus and search


In [ ]:
FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']

flat = pd.read_csv(THES, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
print(f'Thesaurus concepts: {len(flat):,}')

# Precompute: code -> list of ALL-CAPS synonyms (to search against)
VAR_CODE_RE = re.compile(r'^[A-Z0-9]{2,8}$')

def all_caps_syns(synonyms: str) -> set[str]:
    parts = [s.strip() for s in synonyms.split('|') if s.strip()]
    return {p for p in parts if VAR_CODE_RE.fullmatch(p)}

# Invert: ALL-CAPS token -> set of concept codes carrying it as a synonym
token_to_codes: dict[str, list[int]] = {}
for idx, syns in enumerate(flat['synonyms']):
    for tok in all_caps_syns(syns):
        token_to_codes.setdefault(tok, []).append(idx)
print(f'Distinct ALL-CAPS synonym tokens indexed: {len(token_to_codes):,}')


## Match each missing variable, apply disambiguation


In [ ]:
CDASH_SUBSET = 'CDISC CDASH Variable Terminology'
SEND_SUBSET  = 'CDISC SEND Variable Terminology'
CODELIST_RE  = re.compile(r'^CDISC .* Terminology$')

def subset_label(membership: str) -> str:
    in_cdash = CDASH_SUBSET in membership
    in_send  = SEND_SUBSET  in membership
    if in_cdash and in_send: return 'CDASH;SEND'
    if in_cdash: return 'CDASH'
    if in_send:  return 'SEND'
    return 'Other'

pass_b_rows = []
unresolved = []

for var in missing:
    idxs = token_to_codes.get(var, [])
    hits = flat.iloc[idxs] if idxs else flat.iloc[0:0]

    # Exclude codelist-container concepts
    pt = hits['synonyms'].map(lambda s: s.split('|')[0].strip())
    keep = ~pt.str.match(CODELIST_RE, na=False)
    hits = hits[keep].copy()
    hits['preferred_term'] = hits['synonyms'].map(lambda s: s.split('|')[0].strip())

    if len(hits) == 0:
        unresolved.append(var)
        continue

    if len(hits) == 1:
        r = hits.iloc[0]
        pass_b_rows.append({
            'variable':           var,
            'nci_code':           r['code'],
            'natural_name':       r['preferred_term'],
            'definition':         r['definition'],
            'subset':             subset_label(r['subset_membership']),
            'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
            'notes':              '',
        })
    else:
        # Multiple survive — prefer concepts in any CDISC-scoped subset
        cdisc_hits = hits[hits['subset_membership'].str.contains('CDISC ', regex=False)]
        if len(cdisc_hits) == 1:
            r = cdisc_hits.iloc[0]
            dropped = [c for c in hits['code'] if c != r['code']]
            pass_b_rows.append({
                'variable':           var,
                'nci_code':           r['code'],
                'natural_name':       r['preferred_term'],
                'definition':         r['definition'],
                'subset':             subset_label(r['subset_membership']),
                'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
                'notes':              f'Multiple Thesaurus hits {list(hits["code"])}; chose CDISC-scoped {r["code"]} (dropped: {dropped})',
            })
        else:
            # Keep all remaining as separate rows, each flagged
            for _, r in hits.iterrows():
                pass_b_rows.append({
                    'variable':           var,
                    'nci_code':           r['code'],
                    'natural_name':       r['preferred_term'],
                    'definition':         r['definition'],
                    'subset':             subset_label(r['subset_membership']),
                    'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
                    'notes':              f'Ambiguous: {len(hits)} Thesaurus concepts match synonym {var}; manual pick needed',
                })

pass_b = pd.DataFrame(pass_b_rows)
print(f'Pass B rows built:                    {len(pass_b)}')
print(f'Missing vars now resolved:            {pass_b["variable"].nunique() if len(pass_b) else 0}')
print(f'Missing vars still unresolved:        {len(unresolved)}')
print()
if len(pass_b):
    print('Subset distribution:')
    print(pass_b['subset'].value_counts().to_string())
    print()
    print('Flagged Pass B rows:')
    for _, r in pass_b[pass_b['notes'] != ''].iterrows():
        print(f'  {r["variable"]:10} {r["nci_code"]:10} {r["natural_name"][:40]:40} | {r["notes"][:80]}')

print()
print(f'Unresolved ({len(unresolved)}):')
print('  ' + ', '.join(unresolved))


## Merge Pass A + Pass B and rewrite xlsx

Pass A rows preserved byte-for-byte. Pass B rows appended. Same file path.
README sheet regenerated with updated counts and a new section documenting
Pass B.


In [ ]:
pass_a_std = pass_a.rename(columns={
    'Variable':           'variable',
    'NCIt_Code':          'nci_code',
    'Natural_Name':       'natural_name',
    'Definition':         'definition',
    'Subset':             'subset',
    'Applicable_Domains': 'applicable_domains',
    'Notes':              'notes',
})[['variable', 'nci_code', 'natural_name', 'definition', 'subset', 'applicable_domains', 'notes']]

combined = pd.concat([pass_a_std, pass_b], ignore_index=True)

# Sort: empty variable last, otherwise by variable then nci_code
combined['_sort_key'] = combined['variable'].where(combined['variable'] != '', 'ZZZZ')
combined = combined.sort_values(['_sort_key', 'nci_code']).drop(columns='_sort_key').reset_index(drop=True)

print(f'Combined rows: {len(combined):,}')
print(f'  Pass A: {len(pass_a_std):,}')
print(f'  Pass B: {len(pass_b):,}')
print()
print('Subset value counts:')
print(combined['subset'].value_counts().to_string())


In [ ]:
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment

GREEN_HEADER  = PatternFill(start_color='548235', end_color='548235', fill_type='solid')
GREEN_FILL    = PatternFill(start_color='EBF7EE', end_color='EBF7EE', fill_type='solid')
README_HEADER = PatternFill(start_color='595959', end_color='595959', fill_type='solid')
NOTE_FILL     = PatternFill(start_color='FFF3CD', end_color='FFF3CD', fill_type='solid')
HEADER_WHITE  = Font(bold=True, color='FFFFFF')
thin_border = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999'),
)

COLUMNS = [
    ('variable',           'Variable'),
    ('nci_code',           'NCIt_Code'),
    ('natural_name',       'Natural_Name'),
    ('definition',         'Definition'),
    ('subset',             'Subset'),
    ('applicable_domains', 'Applicable_Domains'),
    ('notes',              'Notes'),
]

wb = Workbook()

# README — single-column with grey header band on A1
ws_r = wb.active
ws_r.title = 'README'

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
n_total       = len(combined)
n_a           = len(pass_a_std)
n_b           = len(pass_b)
n_notes       = (combined['notes']    != '').sum()
cosmos_cov    = (combined.loc[combined['applicable_domains'] != '', 'variable']).nunique()
subset_counts = combined['subset'].value_counts()

def subset_line(label):
    return f'    {label:18} {subset_counts.get(label, 0):>6,}'

readme = [
    'SDTM Variable Identity',
    '',
    'Provenance.',
    f'  Generated:        {now}',
    '  Pipeline:         sdtm-narrative/notebooks/10_build_variable_identity.ipynb (Pass A)',
    '                    sdtm-narrative/notebooks/12_variable_identity_bis.ipynb  (Pass B)',
    '  NCIt source:      Thesaurus.txt (NCI EVS FLAT file, versionless)',
    '  COSMoS source:    cosmos-graph/interim/COSMoS_Graph.xlsx',
    '',
    'Scope.',
    f'  Total rows:          {n_total:,}',
    f'    Pass A:            {n_a:,}  (CDISC Variable + Root Variable Terminology subsets)',
    f'    Pass B:            {n_b:,}  (second-pass lookup for 111 missing COSMoS variables)',
    '',
    '  Subset distribution:',
    subset_line('Variable'),
    subset_line('Root'),
    subset_line('Variable+Root'),
    subset_line('CDASH'),
    subset_line('SEND'),
    subset_line('CDASH;SEND'),
    subset_line('Other'),
    '',
    f'  COSMoS variable coverage:  {cosmos_cov}/{len(cosmos_variables)} = {100*cosmos_cov/len(cosmos_variables):.0f}%',
    f'  Flagged in Notes:          {n_notes}',
    '',
    'Columns.',
    '  Variable           SDTM variable code. Root-subset variables use the two-dash '
    'prefix form in SDTMIG usage (e.g. --DESC, --CAT); Thesaurus stores the bare form '
    '(DESC, CAT).',
    '  NCIt_Code          NCIt concept C-code.',
    '  Natural_Name       NCIt preferred term.',
    '  Definition         NCIt definition.',
    '  Subset             CDISC subset the concept belongs to:',
    '                       Variable / Root / Variable+Root  = Pass A primary subsets',
    '                       CDASH / SEND / CDASH;SEND         = Pass B CDISC Variable Terminology family',
    '                       Other                             = Pass B, no CDISC Variable Terminology subset',
    '  Applicable_Domains Semicolon-separated SDTM domain codes where the variable '
    'appears in COSMoS Graph. Empty = not used in COSMoS.',
    '  Notes              Ambiguity flag. Empty = clean extraction. Populated rows are '
    'highlighted in amber in the Variable_Identity sheet.',
    '',
    'Pass B design.',
    '  Pass B was added because 111 COSMoS variables had no entry in the CDISC Variable '
    'Terminology / Root Variable Terminology subsets. Pass B widens the search to the '
    'full Thesaurus by ALL-CAPS synonym match, applies two filters:',
    '    - Excludes concepts whose preferred term matches "CDISC * Terminology" '
    '(codelist containers, not variable concepts).',
    '    - When multiple concepts survive, prefers CDISC-scoped concepts; if still '
    'ambiguous, keeps all rows and flags in Notes.',
    '  Remaining unresolved COSMoS variables are concepts absent from Thesaurus as '
    'ALL-CAPS synonyms (e.g. ETHNIC, FTDTC, BECAT). They are not appended.',
    '',
    'Header colour convention. Green (#548235) = identity layer. Sibling of '
    'SDTM_Test_Identity.xlsx and SDTM_Instrument_Identity.xlsx.',
]
for i, text in enumerate(readme, start=1):
    ws_r.cell(row=i, column=1, value=text)
ws_r['A1'].font = HEADER_WHITE
ws_r['A1'].fill = README_HEADER
ws_r['A1'].alignment = Alignment(vertical='top')
ws_r.column_dimensions['A'].width = 80

# Variable_Identity sheet — green canonical
ws = wb.create_sheet('Variable_Identity')

for j, (field, header) in enumerate(COLUMNS, start=1):
    c = ws.cell(row=1, column=j, value=header)
    c.font = HEADER_WHITE
    c.fill = GREEN_HEADER
    c.border = thin_border
    c.alignment = Alignment(wrap_text=True, vertical='top')

for i, row in enumerate(combined.itertuples(index=False), start=2):
    for j, (field, _) in enumerate(COLUMNS, start=1):
        value = getattr(row, field)
        c = ws.cell(row=i, column=j, value=value)
        c.border = thin_border
        c.alignment = Alignment(wrap_text=True, vertical='top')
        c.fill = NOTE_FILL if (field == 'notes' and value) else GREEN_FILL

# Auto-widths (capped at 50 per canonical; wrap_text handles overflow)
for col in ws.columns:
    width = max((len(str(c.value)) for c in col if c.value is not None), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(width + 2, 50)

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

wb.save(IDENTITY)
print(f'Wrote {IDENTITY}')
print(f'Size: {IDENTITY.stat().st_size / 1024:.0f} KB')


## Sanity checks


In [ ]:
assert combined['nci_code'].is_unique, 'duplicate nci_code'
print('OK: nci_code unique')

still_missing = cosmos_variables - set(combined.loc[combined['variable'] != '', 'variable'])
print(f'COSMoS variables still missing from identity: {len(still_missing)}')
if still_missing:
    for v in sorted(still_missing):
        print(f'  {v}')
